# Data Cleaning — BRFSS 2015 (`_MICHD`)

Two stages:
1. **Stateless cleaning** (`preprocess.py`) — codebook rules, safe to apply to train + test at once
2. **Learned cleaning** (`fit` / `transform`) — imputation, NaN indicators, clipping, one-hot encoding, standardisation — fit on train only

Output: `data/clean/*.npy` ready for `run.ipynb`

In [1]:
import numpy as np
import os
from helpers import load_csv_data, create_csv_submission
from preprocess import stateless_clean, leftover_codes

## 1. Load raw data

In [2]:
RAW_DIR   = "data/raw/"
CLEAN_DIR = "data/clean/"

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(RAW_DIR)

# Read column names from the CSV header
with open(os.path.join(RAW_DIR, "x_train.csv")) as f:
    header = f.readline().strip().split(",")
full_headers = header[1:] if len(header) == x_train.shape[1] + 1 else header

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print(f"Positives: {(y_train == 1).sum()}  Negatives: {(y_train == -1).sum()}")

x_train: (328135, 321)
x_test : (109379, 321)
Positives: 28975  Negatives: 299160


## 2. Stateless cleaning (codebook rules)

All rules come from `preprocess.py` — nothing is learned from the data here.

In [3]:
x_tr_c, clean_names, rep = stateless_clean(x_train, full_headers)
x_te_c, te_names, _     = stateless_clean(x_test,  full_headers)

assert clean_names == te_names, "train and test columns differ after cleaning"

idx = {n: j for j, n in enumerate(clean_names)}

print(f"Columns: {x_train.shape[1]} → {x_tr_c.shape[1]}  (dropped {x_train.shape[1] - x_tr_c.shape[1]})")
print(f"NaN share: {np.isnan(x_train).mean():.3f} → {np.isnan(x_tr_c).mean():.3f}")
print("Rules not found:", rep["rules_not_found"])
print("Unverified NaN share:", rep["unverified_nan_share"])

left = leftover_codes(x_tr_c, clean_names)
print("\nLeftover special codes in mostly-filled columns:")
for item in left:
    print(" ", item)
if not left:
    print("  none")

Columns: 321 → 275  (dropped 46)
NaN share: 0.448 → 0.449
Rules not found: []
Unverified NaN share: {'MAXVO2_': 0.012, 'FC60_': 0.012, 'DROCDY3_': 0.0487}

Leftover special codes in mostly-filled columns:
  ('_AGE80', 77, 0.013)


## 3. Learned cleaning: `fit` / `transform`

Everything estimated from data (medians, quantiles, one-hot categories) is fit on the **training set only**, then applied to both train and test.

In [4]:
# Nominal columns: numeric code has no order → one-hot encode
DEFAULT_ONEHOT = [
    "_STATE", "_RACE", "_PRACE1", "_MRACE1", "_CRACE1", "_RACEGR3",
    "_ASTHMS1", "_CHOLCHK", "_PAREC1", "_LMTACT1", "_LMTWRK1",
    "_LMTSCL1", "MARITAL", "EMPLOY1", "RENTHOM1"
]

def _build(x, p, dtype):
    """Build raw feature matrix (imputed, flagged, clipped, one-hot) before scaling."""
    ix = {n: j for j, n in enumerate(p["in_names"])}
    cols, names = [], []
    for n in p["num"]:
        c = x[:, ix[n]].astype(np.float64)
        miss = np.isnan(c)
        if n in p["flag"]:
            cols.append(miss.astype(dtype))
            names.append(n + "_isnan")
        c[miss] = p["median"][n]
        if n in p["clip"]:
            c = np.clip(c, *p["clip"][n])
        cols.append(c.astype(dtype))
        names.append(n)
    for n, cats in p["onehot"].items():
        c = np.where(np.isnan(x[:, ix[n]]), -1.0, x[:, ix[n]])
        for k in cats:
            cols.append((c == k).astype(dtype))
            names.append(f"{n}={k:g}")
    return np.column_stack(cols), names


def fit(x, names, onehot=DEFAULT_ONEHOT, max_nan=0.8, flag_nan=0.05,
        clip_q=(0.5, 99.5), min_unique_clip=10, dtype=np.float32):
    """Learn all statistics from x (already stateless-cleaned)."""
    names = list(names)
    ix = {n: j for j, n in enumerate(names)}
    p = {"in_names": names, "onehot": {}, "num": [], "median": {},
         "flag": [], "clip": {}, "dtype": dtype}

    for n in onehot:
        if n in ix:
            c = np.where(np.isnan(x[:, ix[n]]), -1.0, x[:, ix[n]])
            p["onehot"][n] = np.unique(c)

    for n in names:
        if n in p["onehot"]:
            continue
        c = x[:, ix[n]]
        miss = np.isnan(c)
        if miss.mean() > max_nan:
            continue
        obs = c[~miss]
        if obs.size == 0 or obs.min() == obs.max():
            continue
        p["num"].append(n)
        p["median"][n] = float(np.median(obs))
        if miss.mean() >= flag_nan:
            p["flag"].append(n)
        if np.unique(obs).size > min_unique_clip:
            p["clip"][n] = tuple(float(v) for v in np.percentile(obs, clip_q))

    raw, feat = _build(x, p, dtype)
    mu = raw.mean(axis=0, dtype=np.float64)
    sd = raw.std(axis=0, dtype=np.float64)
    keep = sd > 1e-8
    p["mu"], p["sd"], p["keep"] = mu[keep], sd[keep], keep
    p["names"] = ["bias"] + [n for n, k in zip(feat, keep) if k]
    return p


def transform(x, p):
    """Apply frozen statistics. Returns (X with bias column in position 0, names)."""
    if x.shape[1] != len(p["in_names"]):
        raise ValueError(f"x has {x.shape[1]} columns, fit saw {len(p['in_names'])}")
    raw, _ = _build(x, p, p["dtype"])
    X = ((raw[:, p["keep"]] - p["mu"]) / p["sd"]).astype(p["dtype"])
    return np.c_[np.ones(len(X), dtype=p["dtype"]), X], p["names"]

## 4. Sanity check with one fold

In [5]:
n = len(y_train)
rng = np.random.default_rng(42)
perm = rng.permutation(n)
tr_idx = perm[:int(0.8 * n)]
va_idx = perm[int(0.8 * n):]

params = fit(x_tr_c[tr_idx], clean_names)
X_tr, feat_names = transform(x_tr_c[tr_idx], params)
X_va, _          = transform(x_tr_c[va_idx], params)

print("X_train:", X_tr.shape, "  dtype:", X_tr.dtype)
print("X_val  :", X_va.shape)
print("NaN in train:", np.isnan(X_tr).sum(), " val:", np.isnan(X_va).sum())
print(f"Features: {len(feat_names)}  (first 5: {feat_names[:5]})")

X_train: (262508, 369)   dtype: float32
X_val  : (65627, 369)
NaN in train: 0  val: 0
Features: 369  (first 5: ['bias', 'NUMADULT_isnan', 'NUMADULT', 'NUMMEN_isnan', 'NUMMEN'])


## 5. Save clean data

In [6]:
os.makedirs(CLEAN_DIR, exist_ok=True)

np.save(CLEAN_DIR + "x_train_clean.npy", x_tr_c)
np.save(CLEAN_DIR + "x_test_clean.npy",  x_te_c)
np.save(CLEAN_DIR + "y_train.npy",        y_train)
np.save(CLEAN_DIR + "train_ids.npy",      train_ids)
np.save(CLEAN_DIR + "test_ids.npy",       test_ids)

with open(CLEAN_DIR + "columns.txt", "w") as f:
    f.write("\n".join(clean_names))

print("Saved to", CLEAN_DIR)
print("x_train_clean:", x_tr_c.shape)
print("x_test_clean :", x_te_c.shape)

Saved to data/clean/
x_train_clean: (328135, 275)
x_test_clean : (109379, 275)
